# Notebook 03: Cross-Encoder Reranking

**Goal:** Implement cross-encoder reranking (`cross-encoder/ms-marco-MiniLM-L-6-v2`) on top of BM25 candidates to refine retrieval accuracy and measure CPU inference latency.

## Objectives:
1. Load `ms-marco-MiniLM-L-6-v2` cross-encoder model
2. Fetch top-10 candidates from BM25 sparse retriever
3. Re-score candidates using deep cross-attention semantic alignment
4. Measure latency on CPU (<500ms limit)
5. Run verification tests for semantic reranking precision

In [1]:
# Cell 1: Import dependencies
import os
import sys
import time
from typing import List, Dict, Any, Tuple

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sentence_transformers import CrossEncoder
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from scripts.create_sample_pdf import create_sample_pdf
import pdfplumber
import re
import nltk

print("Notebook 03 dependencies loaded successfully.")

C:\Users\kesha\AppData\Local\Temp\ipykernel_42592\3674929851.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Notebook 03 dependencies loaded successfully.


In [2]:
# Cell 2: Initialize BM25 and CrossEncoder Model
def get_sample_chunks() -> List[Document]:
    pdf_path = os.path.join(project_root, "data", "sample_ai_report.pdf")
    if not os.path.exists(pdf_path):
        create_sample_pdf(pdf_path)
    raw_pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw_pages.append(page.extract_text() or "")
    full_text = "\n\n".join(raw_pages)
    section_splits = re.split(r'(?m)^(?:(?=\d+\.\s+[A-Z]))', full_text)
    docs = []
    chunk_id = 0
    for split in section_splits:
        split_clean = split.strip()
        if not split_clean:
            continue
        lines = split_clean.split('\n')
        sec_title = lines[0].strip() if re.match(r'^\d+\.\s+[A-Z]', lines[0].strip()) else "Document Title"
        body_lines = lines[1:] if re.match(r'^\d+\.\s+[A-Z]', lines[0].strip()) else lines
        paragraphs = [re.sub(r'\s+', ' ', p).strip() for p in "\n".join(body_lines).split('\n\n') if p.strip()]
        for p_idx, para in enumerate(paragraphs):
            if not para.endswith(('.', '!', '?', ':', ';')):
                para += '.'
            chunk_id += 1
            docs.append(Document(
                page_content=para,
                metadata={"chunk_id": f"chunk_{chunk_id:03d}", "section": sec_title}
            ))
    return docs

docs = get_sample_chunks()
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 10

reranker_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print(f"Loading CrossEncoder model: {reranker_model_name}...")
reranker = CrossEncoder(reranker_model_name)
print("CrossEncoder model loaded successfully.")

Loading CrossEncoder model: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

CrossEncoder model loaded successfully.


In [3]:
# Cell 3: Reranking Function & Latency Measurement
def rerank_documents(query: str, top_k_bm25: int = 10, top_n_reranked: int = 3) -> Tuple[List[Dict[str, Any]], float]:
    """Retrieve candidates via BM25 and rerank using CrossEncoder."""
    start_time = time.time()
    
    # 1. Sparse BM25 retrieval
    bm25_retriever.k = top_k_bm25
    candidates = bm25_retriever.invoke(query)
    
    if not candidates:
        return [], (time.time() - start_time) * 1000
        
    # 2. Pair query with each candidate passage
    pairs = [[query, doc.page_content] for doc in candidates]
    
    # 3. Cross-encoder scoring
    scores = reranker.predict(pairs)
    
    # 4. Combine and sort by score descending
    scored_docs = []
    for doc, score in zip(candidates, scores):
        scored_docs.append({
            "doc": doc,
            "score": float(score),
            "chunk_id": doc.metadata["chunk_id"],
            "section": doc.metadata["section"]
        })
        
    scored_docs.sort(key=lambda x: x["score"], reverse=True)
    latency_ms = (time.time() - start_time) * 1000
    return scored_docs[:top_n_reranked], latency_ms

query = "How fast does sparse retrieval run on 50 page documents?"
reranked_results, latency = rerank_documents(query)
print(f"Query: '{query}'")
print(f"Total Pipeline Latency (BM25 + Rerank): {latency:.2f} ms")
for idx, item in enumerate(reranked_results):
    print(f"Rank {idx+1} | Score: {item['score']:.4f} | {item['chunk_id']} | {item['doc'].page_content[:100]}...")

Query: 'How fast does sparse retrieval run on 50 page documents?'
Total Pipeline Latency (BM25 + Rerank): 323.76 ms
Rank 1 | Score: 4.1888 | chunk_005 | The average CPU latency for BM25 retrieval across a 50-page document is under 15 milliseconds. Cross...
Rank 2 | Score: -0.9618 | chunk_003 | Standard dense retrieval pipelines rely heavily on vector embeddings and high-dimensional indexes, w...
Rank 3 | Score: -2.6281 | chunk_002 | ConverseIQ is an offline-first, voice-enabled document intelligence platform designed for privacy-pr...


In [4]:
# Cell 4: Paraphrased Query Precision Test
para_query = "What hardware specs are required for local RAG execution?"
para_results, para_latency = rerank_documents(para_query)
print(f"Paraphrased Query: '{para_query}'")
print(f"Latency: {para_latency:.2f} ms")
for idx, item in enumerate(para_results):
    print(f"Rank {idx+1} | Score: {item['score']:.4f} | {item['chunk_id']} | {item['section']}")

Paraphrased Query: 'What hardware specs are required for local RAG execution?'
Latency: 70.83 ms
Rank 1 | Score: -10.8133 | chunk_004 | 3. Local LLM Integration and Memory Management
Rank 2 | Score: -10.8591 | chunk_003 | 2. Hierarchical Retrieval Architecture
Rank 3 | Score: -10.8592 | chunk_005 | 4. System Performance and Resource Requirements


In [5]:
# Cell 5: Verification Assertions for Notebook 03
print("Running verification assertions on Cross-Encoder Reranking...")
assert len(reranked_results) > 0, "Verification Failed: No reranked results returned!"
assert latency < 500.0, f"Verification Failed: Reranking latency {latency:.2f} ms exceeds 500ms threshold!"
assert reranked_results[0]["score"] >= reranked_results[-1]["score"], "Verification Failed: Results are not sorted by score descending!"
print("✅ ALL VERIFICATION TESTS PASSED FOR NOTEBOOK 03!")

Running verification assertions on Cross-Encoder Reranking...
✅ ALL VERIFICATION TESTS PASSED FOR NOTEBOOK 03!
